# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima12aa/fa-ml/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [10]:
# --- 1. Token + connection ---
import os
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected.")

Connected.


In [11]:
# --- 2. February features (with client_hash_id) ---
feb_impressions = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) AS feb_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

feb_ctr = con.sql(f"""
    SELECT content_hash_id,
        SUM(gsc_clicks) AS feb_clicks,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
             ELSE NULL END AS feb_ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_position = con.sql(f"""
    SELECT content_hash_id, AVG(gsc_avg_position) AS feb_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE AND gsc_avg_position > 0
    GROUP BY content_hash_id
""").df()

content_features = con.sql(f"""
    SELECT content_hash_id, word_count, search_volume
    FROM {TABLES['dim_content']}
""").df()

print("feb_impressions:", feb_impressions.shape)
print("feb_ctr:", feb_ctr.shape)
print("feb_position:", feb_position.shape)
print("content_features:", content_features.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feb_impressions: (153559, 3)
feb_ctr: (153559, 3)
feb_position: (151956, 2)
content_features: (519606, 3)


In [12]:
# --- 3. March impressions + label ---
march_impressions = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS march_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

trend_data = feb_impressions.merge(march_impressions, on="content_hash_id", how="inner")
trend_data["is_declining"] = (
    trend_data["march_impressions"] < 0.8 * trend_data["feb_impressions"]
).astype(int)

print("trend_data:", trend_data.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

trend_data: (134238, 5)


In [13]:
# --- 4. Assemble final training table ---
training_table = content_features \
    .merge(feb_impressions, on="content_hash_id", how="inner") \
    .merge(feb_ctr, on="content_hash_id", how="left") \
    .merge(feb_position, on="content_hash_id", how="left") \
    .merge(trend_data[["content_hash_id", "is_declining"]], on="content_hash_id", how="inner")

feature_cols = ["word_count", "search_volume", "feb_impressions", "feb_ctr", "feb_avg_position"]
training_table_clean = training_table.dropna(subset=feature_cols)

print("training_table_clean:", training_table_clean.shape)

training_table_clean: (75189, 9)


In [14]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

verified_features = ["feb_ctr", "feb_avg_position"]
X_v = training_table_clean[verified_features]
y_v = training_table_clean["is_declining"]
groups_v = training_table_clean["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X_v, y_v, groups=groups_v))

X_tr, X_te = X_v.iloc[train_idx], X_v.iloc[test_idx]
y_tr, y_te = y_v.iloc[train_idx], y_v.iloc[test_idx]

forest_v = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
scores_v = forest_v.predict_proba(X_te)[:, 1]

print("=== Random Forest, verified features only (CTR + position) ===")
print("Precision@20:", precision_at_k(scores_v, y_te.values, 20))
print("Precision@50:", precision_at_k(scores_v, y_te.values, 50))
print()
print("Compare to baseline rule (w04): P@20=0.15, P@50=0.22")
print("Compare to full 5-feature model (w05): P@20=0.15, P@50=0.22")

=== Random Forest, verified features only (CTR + position) ===
Precision@20: 0.25
Precision@50: 0.32

Compare to baseline rule (w04): P@20=0.15, P@50=0.22
Compare to full 5-feature model (w05): P@20=0.15, P@50=0.22


In [15]:
# Try multiple random splits, see if the improvement holds consistently
results = []
for seed in [1, 2, 3, 42, 100]:
    gss_test = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr_idx, te_idx = next(gss_test.split(X_v, y_v, groups=groups_v))

    X_tr_t, X_te_t = X_v.iloc[tr_idx], X_v.iloc[te_idx]
    y_tr_t, y_te_t = y_v.iloc[tr_idx], y_v.iloc[te_idx]

    m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_t, y_tr_t)
    s = m.predict_proba(X_te_t)[:, 1]

    results.append({
        "seed": seed,
        "p_at_20": precision_at_k(s, y_te_t.values, 20),
        "p_at_50": precision_at_k(s, y_te_t.values, 50),
        "test_clients": len(set(training_table_clean.iloc[te_idx]["client_hash_id"]))
    })

import pandas as pd
print(pd.DataFrame(results))

   seed  p_at_20  p_at_50  test_clients
0     1     0.25     0.26            10
1     2     0.30     0.22            10
2     3     0.35     0.32            10
3    42     0.25     0.32            10
4   100     0.30     0.16            10


In [16]:
# Build the ranked queue using the verified-features model's predicted probabilities
queue_data = training_table_clean.copy()

# Use the trained model to score ALL pages (not just the test set) for the final playbook
all_scores = forest_v.predict_proba(queue_data[verified_features])[:, 1]
queue_data["decline_risk_score"] = all_scores

# Reason code + action, same style as w04
queue_data["reason_code"] = "model_flagged_ctr_position_risk"
queue_data["action"] = "review_title_meta"

ranked_playbook_queue = queue_data.sort_values("decline_risk_score", ascending=False)
print(ranked_playbook_queue[["content_hash_id", "feb_avg_position", "feb_ctr", "decline_risk_score", "reason_code", "action"]].head(10))

                 content_hash_id  feb_avg_position  feb_ctr  \
116300  content_faf47c5e21b4a1a3         14.393681      0.0   
12191   content_ad1edb0cd05fa694         14.392857      0.0   
2167    content_859534508846037c         78.708333      0.0   
129659  content_f8a4b53bd01ee58e         78.750000      0.0   
29241   content_9054d8e25d4162f0         18.226852      0.0   
41366   content_712977d9703c59f3         18.224603      0.0   
65415   content_dbd29b9780455dc0         78.692308      0.0   
2347    content_94fc5de23263e329         78.750000      0.0   
50451   content_57032b3249d7084d         14.392247      0.0   
367     content_0563a617dfe608f4         18.228387      0.0   

        decline_risk_score                      reason_code             action  
116300               0.995  model_flagged_ctr_position_risk  review_title_meta  
12191                0.995  model_flagged_ctr_position_risk  review_title_meta  
2167                 0.995  model_flagged_ctr_position_risk  re

In [17]:
# ============================================================
# ISSUE FOUND: the raw model output doesn't match our verified signal
# ------------------------------------------------------------
# Our Signal 1 (w04, CONFIRMED) specifically diagnosed: GOOD position
# (<=10) + surprisingly LOW CTR = a listing/title problem, since the
# page already earns visibility but isn't converting clicks.
#
# But the trained model (CTR + position only, no gate logic) learned
# its own pattern from training data -- and its top-ranked pages turned
# out to have position 14-79 with CTR=0. A page ranked #78 getting zero
# clicks is EXPECTED (nobody scrolls that deep), not diagnostic of a
# listing problem -- it's a visibility/content problem instead, a
# different diagnosis than what our reason code and action
# ("review_title_meta") actually claims.
#
# WHY THIS HAPPENED: the model wasn't told our specific "good position +
# low CTR" theory -- it just found whatever correlation reduced error on
# training data, which turned out to be "very poor position + low CTR
# correlates with a page never having ranked well in the first place",
# not our original title-fix diagnosis.
#
# FIX: filter the model's output back down to only pages that ALSO
# satisfy our original verified condition (position <= 10), so the
# queue stays consistent with the diagnosis our reason code actually
# claims.
# ============================================================

playbook_queue_filtered = ranked_playbook_queue[
    ranked_playbook_queue["feb_avg_position"] <= 10
].reset_index(drop=True)

print("Qualifying pages (position <= 10):", len(playbook_queue_filtered))
playbook_queue_filtered[["content_hash_id", "feb_avg_position", "feb_ctr", "decline_risk_score", "reason_code", "action"]].head(10)

Qualifying pages (position <= 10): 46365


,content_hash_id,feb_avg_position,feb_ctr,decline_risk_score,reason_code,action
0,content_16b8d1639735c69b,9.302862,0.0,0.995,model_flagged_ctr_position_risk,review_title_meta
1,content_2500405af95cd0ce,9.259344,0.0,0.990,model_flagged_ctr_position_risk,review_title_meta
2,content_67ec5636bbad1d32,7.194156,0.0,0.990,model_flagged_ctr_position_risk,review_title_meta
3,content_9b0bae09f519e4ab,9.259637,0.0,0.990,model_flagged_ctr_position_risk,review_title_meta
4,content_505c506b3c53763c,9.045455,0.0,0.985,model_flagged_ctr_position_risk,review_title_meta
5,content_7567f6c0c2a1d48f,9.045455,0.0,0.985,model_flagged_ctr_position_risk,review_title_meta
6,content_27590efd5d682d67,9.230769,0.0,0.985,model_flagged_ctr_position_risk,review_title_meta
7,content_6d907e7da69cd0ce,7.194487,0.0,0.980,model_flagged_ctr_position_risk,review_title_meta
8,content_8bc34345c9fb881b,7.194444,0.0,0.980,model_flagged_ctr_position_risk,review_title_meta
9,content_0d6b8497c66f06ca,9.303354,0.0,0.980,model_flagged_ctr_position_risk,review_title_meta


(the text below is from claude but the ideas are mine; i ask it to ask questions which i answer and claude refines and rewrites)

## Ranked actions + reason codes

**The queue:** We use a Random Forest model trained on our two verified signals (February CTR
and average position), scored and ranked by predicted decline risk, filtered to pages ranking
well (position ≤ 10) — matching our Week 4 CONFIRMED finding that good position paired with
unexpectedly low CTR signals a listing problem, not a content problem.

**Reason code:** `model_flagged_ctr_position_risk` — assigned when a page ranks in the top 10
positions but the model's predicted decline risk (based on CTR and position) is high.

**Action:** `review_title_meta` — review and likely rewrite the page's title/meta description,
since the page already earns visibility but isn't converting it into clicks.

**Why this beats our Week 4 hand rule:** Restricting the model to only our two verified
signals (rather than all 5 features) improved Precision@20 from 0.15 to 0.25-0.35 (varying
somewhat by split, per our stability check) and Precision@50 from 0.22 to 0.16-0.32 — a real,
though not perfectly stable, improvement over the hand-written baseline.

**A decay/refresh insight (external, not independently verified by us):** FlyRank's own
research paper (March 2026) found that old, stale content which gets refreshed shows
substantial performance gains in their portfolio — though the paper itself notes the exact
sample size for this specific claim isn't disclosed (a methodology gap we identified in our
Week 6 audit). We treat this as a directional insight worth incorporating into future refresh
scheduling, not as a verified finding from our own data — our own attempt to build a
staleness signal (Week 4) was inconclusive due to data availability issues.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.